# Wave 2 — Account 2: LLaVA-1.5-7B Open-Ended (Gaussian + SwapBlend)

**Reliance-Utility Extension Phase**

- Model: `llava-hf/llava-1.5-7b-hf`
- Dataset: A-OKVQA (N=500, `main_ids.json`)
- Corruptions: `gaussian`, `swapblend` (2 only — fits 12h ceiling at ~9.6h)
- Expected output: 40 shards
- Output: `account2_wave2_llava_open.zip` in `/kaggle/working/`
- **Download from the Output tab of your committed Kaggle notebook version.**

> ⚠️ Run with **Save & Run All** (Commit mode), GPU T4 x2, Internet ON.


In [ ]:
import os
import sys
import time
import subprocess

print("=" * 60)
print("WAVE 2 - ACCOUNT 2: LLAVA-1.5-7B OPEN-ENDED (GAUSSIAN + SWAPBLEND)")
print("=" * 60)

# STEP 1: FIX CUDA CONFLICT & INSTALL DEPENDENCIES
print("\n>>> Step 1/5: Fixing CUDA conflict and installing dependencies...")
subprocess.run("pip uninstall -y torchaudio", shell=True)
subprocess.run(
    "pip install -q -U transformers accelerate datasets pillow torchvision bitsandbytes",
    shell=True, check=True
)

# STEP 2: CLONE OR UPDATE REPOSITORY (EXTENSION BRANCH)
print("\n>>> Step 2/5: Synchronizing GitHub repository (extension branch)...")
repo_dir = "/kaggle/working/reliance-utility"
if not os.path.exists(repo_dir):
    subprocess.run(
        f"git clone https://github.com/Stealth768/reliance-utility.git {repo_dir}",
        shell=True, check=True
    )
else:
    print("Repository exists on disk. Resetting and pulling latest...")
    subprocess.run(f"cd {repo_dir} && git reset --hard HEAD", shell=True, check=True)

subprocess.run(
    f"cd {repo_dir} && git fetch --all --tags && git checkout extension && git pull origin extension",
    shell=True, check=True
)

import torch
assert torch.cuda.is_available(), "GPU accelerator not detected!"
print(f"CUDA GPU Verified: {torch.cuda.get_device_name(0)}")

# STEP 3: PRE-FLIGHT CHECK GUARD
print("\n>>> Step 3/5: Running mandatory pre-flight check...")
check_cmd = f"cd {repo_dir} && python -u scripts/run_vlm_open.py --model llava-hf/llava-1.5-7b-hf --assistant_prefix answer_colon --check"
check_res = subprocess.run(check_cmd, shell=True)
if check_res.returncode != 0:
    print(f"\n[FATAL ERROR] LLaVA-1.5-7B open-ended pre-check failed with exit code {check_res.returncode}!")
    print("Refusing to continue with full production run.")
    sys.exit(check_res.returncode)

# STEP 4: PRODUCTION RUN
print("\n>>> Step 4/5: Running production open-ended LLaVA-1.5-7B...")
out_open = "/kaggle/working/results/raw/vlm_open"
os.makedirs(out_open, exist_ok=True)

# 2 corruptions (gaussian,swapblend) to fit within 12h session ceiling (~9.6h runtime)
prod_cmd = (
    f"cd {repo_dir} && python -u scripts/run_vlm_open.py "
    f"--model llava-hf/llava-1.5-7b-hf "
    f"--assistant_prefix answer_colon "
    f"--corruptions gaussian,swapblend "
    f"--data_ids results/derived/main_ids.json "
    f"--n 500 --seed 0 --shard_size 25 "
    f"--out {out_open}"
)

t_start = time.time()
subprocess.run(prod_cmd, shell=True, check=True)
print(f"\nLLaVA-1.5-7B Open-Ended finished in {(time.time() - t_start)/60:.1f} min!")

# STEP 5: ZIP OUTPUT ARCHIVE
print("\n>>> Step 5/5: Packaging shards into zip...")
zip_name = "account2_wave2_llava_open.zip"
zip_path = f"/kaggle/working/{zip_name}"

subprocess.run(
    f"cd /kaggle/working && zip -r -q {zip_name} results/raw/vlm_open",
    shell=True, check=True
)
file_size_mb = os.path.getsize(zip_path) / (1024 * 1024)
print("=" * 70)
print(f"SUCCESS! Output archive saved to: {zip_path} ({file_size_mb:.2f} MB)")
print("=" * 70)
print("NOTE: Download this archive directly from the 'Output' tab of your committed Kaggle notebook version.")
